In [1]:
# Import libraries and load the CSV
import pandas as pd
import numpy as np

df = pd.read_csv("dirty_cafe_sales.csv")

In [2]:
# Clean column names
df.columns = (
    df.columns
    .str.strip()            # remove extra spaces at the front/back of column names
    .str.lower()            # make all column names lowercase
    .str.replace(" ", "_")  # replace spaces inside column names with underscores
)

In [3]:
# Remove extra spaces in text
df = df.apply(lambda col: col.str.strip() if col.dtype == ("object", "str", "string") else col)

# Meaning:
# - apply() checks each column one by one
# - if the column is text, strip spaces
# - otherwise, return the column unchanged

In [4]:
# Check column names, number of rows, missing values, and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   transaction_id    10000 non-null  str  
 1   item              9667 non-null   str  
 2   quantity          9862 non-null   str  
 3   price_per_unit    9821 non-null   str  
 4   total_spent       9827 non-null   str  
 5   payment_method    7421 non-null   str  
 6   location          6735 non-null   str  
 7   transaction_date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [5]:
# Replace NaN with Unknown
df["item"] = df["item"].fillna("Unknown").replace(("UNKNOWN","ERROR"), "Unknown")
df["item"].value_counts()

item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
Unknown      969
Name: count, dtype: int64

In [6]:
df["payment_method"] = df["payment_method"].fillna("Unknown").replace(("UNKNOWN", "ERROR"), "Unknown")
df["payment_method"].value_counts()

payment_method
Unknown           3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

In [7]:
df["location"] = df["location"].fillna("Unknown").replace(("UNKNOWN", "ERROR"), "Unknown")
df["location"].value_counts()

location
Unknown     3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64

In [8]:
# Price of Cake = 3.0
df[df["item"] == "Cake"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
3.0        1085
NaN          21
ERROR        19
UNKNOWN      14
Name: count, dtype: int64

In [9]:
# Price of Coffee = 2.0
df[df["item"] == "Coffee"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
2.0        1108
UNKNOWN      20
NaN          19
ERROR        18
Name: count, dtype: int64

In [10]:
# Price of Cookie = 1.0
df[df["item"] == "Cookie"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
1.0        1026
NaN          24
UNKNOWN      21
ERROR        21
Name: count, dtype: int64

In [11]:
# Price of Juice = 3.0
df[df["item"] == "Juice"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
3.0        1110
ERROR        26
UNKNOWN      18
NaN          17
Name: count, dtype: int64

In [12]:
# Price of Salad = 5.0
df[df["item"] == "Salad"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
5.0        1082
ERROR        34
UNKNOWN      16
NaN          16
Name: count, dtype: int64

In [13]:
# Price of Sandwich = 4.0
df[df["item"] == "Sandwich"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
4.0        1082
UNKNOWN      19
NaN          17
ERROR        13
Name: count, dtype: int64

In [14]:
# Price of Smoothie = 4.0
df[df["item"] == "Smoothie"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
4.0        1036
NaN          24
ERROR        19
UNKNOWN      17
Name: count, dtype: int64

In [15]:
# Price of Tea = 1.5
df[df["item"] == "Tea"]["price_per_unit"].value_counts(dropna=False)

price_per_unit
1.5        1023
ERROR        25
UNKNOWN      21
NaN          20
Name: count, dtype: int64

In [16]:
df["price_per_unit"] = pd.to_numeric(df["price_per_unit"], errors="coerce")

In [17]:
# Fill missing item prices
df.loc[(df["item"]== "Cake") & (df["price_per_unit"].isna()), "price_per_unit"] = 3.0
df.loc[(df["item"]== "Coffee") & (df["price_per_unit"].isna()), "price_per_unit"] = 2.0
df.loc[(df["item"]== "Cookie") & (df["price_per_unit"].isna()), "price_per_unit"] = 1.0
df.loc[(df["item"]== "Juice") & (df["price_per_unit"].isna()), "price_per_unit"] = 3.0
df.loc[(df["item"]== "Salad") & (df["price_per_unit"].isna()), "price_per_unit"] = 5.0
df.loc[(df["item"]== "Sandwich") & (df["price_per_unit"].isna()), "price_per_unit"] = 4.0
df.loc[(df["item"]== "Smoothie") & (df["price_per_unit"].isna()), "price_per_unit"] = 4.0
df.loc[(df["item"]== "Tea") & (df["price_per_unit"].isna()), "price_per_unit"] = 1.5

In [18]:
# Validate prices after filling
df.groupby("item")["price_per_unit"].value_counts(dropna=False)

item      price_per_unit
Cake      3.0               1139
Coffee    2.0               1165
Cookie    1.0               1092
Juice     3.0               1171
Salad     5.0               1148
Sandwich  4.0               1131
Smoothie  4.0               1096
Tea       1.5               1089
Unknown   3.0                234
          4.0                213
          5.0                122
          2.0                119
          1.0                117
          1.5                110
          NaN                 54
Name: count, dtype: int64

In [19]:
# Fill missing item based on price_per_unit
# We do NOT fill 3.0 because it could be Cake or Juice.
# We do NOT fill 4.0 because it could be Sandwich or Smoothie.
df.loc[(df["item"] == "Unknown") & (df["price_per_unit"] == 1.5), "item"] = "Tea"
df.loc[(df["item"] == "Unknown") & (df["price_per_unit"] == 1.0), "item"] = "Cookie"
df.loc[(df["item"] == "Unknown") & (df["price_per_unit"] == 5.0), "item"] = "Salad"
df.loc[(df["item"] == "Unknown") & (df["price_per_unit"] == 2.0), "item"] = "Coffee"

In [20]:
# Check item-price again
df.groupby("item")["price_per_unit"].value_counts(dropna=False)

item      price_per_unit
Cake      3.0               1139
Coffee    2.0               1284
Cookie    1.0               1209
Juice     3.0               1171
Salad     5.0               1270
Sandwich  4.0               1131
Smoothie  4.0               1096
Tea       1.5               1199
Unknown   3.0                234
          4.0                213
          NaN                 54
Name: count, dtype: int64

In [21]:
# Convert to_numeric, fill with NaN
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["total_spent"] = pd.to_numeric(df["total_spent"], errors="coerce")

# Get total_spent using formula quantity * price_per_unit
df["total_spent"] = df["total_spent"].fillna(df["quantity"] * df["price_per_unit"])

In [22]:
# Get quantity using formula total_spent/price_per_unit
df["quantity"] = df["quantity"].fillna(df["total_spent"]/ df["price_per_unit"])

# Get price_per_unit using formula total_spent/quantity
df["price_per_unit"] = df["price_per_unit"].fillna(df["total_spent"]/ df["quantity"])

In [23]:
# Convert transaction_date into a real datetime column
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")

In [24]:
# Inspect the cleaned dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  str           
 1   item              10000 non-null  str           
 2   quantity          9977 non-null   float64       
 3   price_per_unit    9994 non-null   float64       
 4   total_spent       9977 non-null   float64       
 5   payment_method    10000 non-null  str           
 6   location          10000 non-null  str           
 7   transaction_date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


In [25]:
# Filter rows with unknown
df[
    (df["item"] == "Unknown") |
    (df["payment_method"] == "Unknown") |
    (df["location"] == "Unknown")
]

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Unknown,2023-03-31
6,TXN_4433211,Unknown,3.0,3.0,9.0,Unknown,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,Unknown,2023-10-28
8,TXN_4717867,Unknown,5.0,3.0,15.0,Unknown,Takeaway,2023-07-28
...,...,...,...,...,...,...,...,...
9994,TXN_7851634,Unknown,4.0,4.0,16.0,Unknown,Unknown,2023-01-08
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Unknown,Unknown,2023-08-30
9996,TXN_9659401,Unknown,3.0,1.0,3.0,Digital Wallet,Unknown,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Unknown,2023-03-02


In [26]:
df.to_csv("cleaned_cafe_sales.csv", index=False)